# AIST 2026 — Turkish NLU segmentation (Kaggle runner)

## ⚠️ Do these BEFORE running, or the clone/pip will fail
1. **Verify your Kaggle account by phone** (Profile → Settings → Phone Verification).
   Internet stays locked until you do — this causes `Could not resolve host: github.com`.
2. **Make the GitHub repo Public** (GitHub → repo → Settings → Danger Zone → Change visibility).
3. In the **right-hand Settings panel**: **Accelerator = GPU T4 x2** and **Internet = On**.
   (If the Internet toggle is greyed out, you skipped step 1.)

Note: `%cd` is a Jupyter magic — keep it on its **own line**, never `%cd dir && pip ...`.
The full 3×3×3 matrix fine-tunes base encoders in minutes per cell (~1–3 GPU-h total).

In [ ]:
# 1a) Clone the repo (needs Internet = On)
!git clone https://github.com/SamedAlpARSLAN/TurkishNLU.git

In [ ]:
# 1b) Enter the folder (own line!) and install deps
%cd TurkishNLU
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 2) Sanity + data + mandatory alignment guard (must be ALL PASS)
!python tests/test_core.py
!python scripts/download_data.py
!python scripts/validate_alignment.py --data data/raw/tr-TR.jsonl --models berturk mbert xlmr

In [ ]:
# 3) Training-free analyses for the paper
!python scripts/corpus_stats.py
!python scripts/tokenizer_report.py --reference morphological --scope slot
!python scripts/morphology_compare.py

In [ ]:
# 4) Quick single run to confirm GPU timing, then the full matrix (27 runs)
!python -m src.train --config configs/base.yaml --set model_name=dbmdz/bert-base-turkish-cased segmentation=morphological seed=42
!python scripts/run_matrix.py

In [ ]:
# 5) Tables + figures + significance + morphology-aware error analysis
!python scripts/aggregate.py
!python scripts/make_tables.py
!python scripts/make_figures.py
!python scripts/error_report.py --predictions outputs/berturk_morphological_seed42/test_predictions.json --data data/raw/tr-TR.jsonl
!python scripts/aggregate.py --compare outputs/mbert_morphological_seed42/test_predictions.json outputs/mbert_native_seed42/test_predictions.json

In [ ]:
# 6) Save outputs before the session ends (Kaggle wipes /kaggle/working otherwise)
!zip -r outputs.zip outputs results paper/tables_auto.tex && echo 'Download outputs.zip from the Output tab'